# Multimodal integration of 10x Xenium and Imaging mass cytometry (IMC) visualization in Rakaia

Thie notebook demonstrates how to re-create the Extended Data Figure 1 from the Rakaia manuscript. 

In [1]:
import numpy as np
import pandas as pd
from skimage.transform import warp
from tifffile import imread, imwrite, TiffFile
from numpy.linalg import inv
from rakaia.parsers.spatial import (spatial_canvas_dimensions, spatial_grid_single_marker)
import spatialdata as sd
import rasterio
from rasterio.features import rasterize
import warnings

warnings.filterwarnings('ignore')

/home/admin/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/admin/.local/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


**Before starting**: This notebook assumes that you have performed an alignment of the 10x Xenium DAPI IF morphology image to the IMC ROI `A06_ROI_006.ome.tiff` using [twocan](https://github.com/camlab-bioml/twocan). Information on performing the alignment can be found [here](https://gist.github.com/harrig12/aa2f48220051eed9e28bbcb3567229b4).

To run this notebook, the following data are required in this directory:
- the IMC ROI in tiff format named `A06_ROI_006.ome.tiff`
- The cell-level transcript expression for the 10x Xenium slide in `zarr` format named as `section2_zarr`
- All 4 morphology images for the 10x Xenium run
- The output affine transformation matrix in CSV format from [twocan](https://github.com/camlab-bioml/twocan) named `affine_twocan.csv`

We start with reading in the IMC multi-channel tiff and the 10x Xenium DAPI IF morphology image (designated as `0000` in the 4-channel morphology stack). 

In [2]:
imc_stack = imread("A06_ROI_006.ome.tiff").astype(np.uint32)
with TiffFile("morphology_focus_0000.ome.tif") as tif:
    for page in tif.pages:
        dapi_if = page.asarray().astype(np.uint32)

Next, we will need to modify the twocan transformation matrix to accommodate transforming the Xenium cell expression matrices into the IMC resolution. We will read in the matrix and scale it to the 1 µm Xenium resolution. We will also set `use_affine_inv = False` as the matrix from twocan transforms the IMC image into the IF image. 

In [4]:
twocan_affine = pd.read_csv("affine_twocan.csv", header=None).to_numpy()
dapi_res = 0.2125

affine_for_xenium = np.vstack([(dapi_res * twocan_affine), [0,0,1]])

# # IMP: need to toggle this based on which direction the affine transformation from twocan goes
use_affine_inv = False

We will read the 10x Xenium transcript expression per cell from the`zarr` directory, and compute some coordinate offsets that will be used for visualisation purposes, and when transforming the Xenium markers into the IMC resolution. 

In [18]:
sdata = sd.read_zarr("section2_zarr/")

grid_width, grid_height, x_min, y_min = spatial_canvas_dimensions(sdata.tables['table'])

# # apply the offset to the transform to take into account the trimming of dead space in rakaia
T_offset_inv = np.array([
    [0, 0, -x_min],
    [0, 0, -y_min],
    [0, 0, 0]
])

tform_1 = affine_for_xenium + T_offset_inv


Next, we will convert the 10x Xenium cell segmentation boundary mask into a dense 2D array so that we can transform it and overlay the cell segmentations in Rakaia. This step will output the transformed Xenium segmentation as `xenium_mask_trans_imc_res.tiff` with the same dimensions as the IMC ROI. 

In [19]:
### transform mask ####
flip = True

adata = sdata.tables['table']
cells = sdata.shapes['cell_boundaries']
# get the int bounds to know where the segmentation mask is
x_min, y_min = np.min((adata.obsm['spatial']), axis=0)
x_max, y_max = np.max((adata.obsm['spatial']), axis=0)

transform = rasterio.transform.from_bounds(x_min, y_min, x_max,
                        y_max, int(x_max - x_min), int(y_max - y_min))

shapes = [(geom, idx) for idx, geom in enumerate(cells.geometry)]

# Set the mask shape to be the same as the transcript bounds in rakaia
mask = rasterize(shapes=shapes, out_shape=(int(y_max - y_min), int(x_max - x_min)),
                             transform=transform, dtype='int32')

mask = np.flip(mask, axis=0) if flip else mask

# IMP: need to use order 0 on the mask warp to preserve the integers
xenium_mask_transform = warp(
    image=mask,
    inverse_map=inv(tform_1) if use_affine_inv else tform_1,
    output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
    order=0,
    mode='constant',
    cval=0.0,
    preserve_range=True
).astype(np.uint32)


imwrite("./xenium_mask_trans_imc_res.tiff",
        xenium_mask_transform, photometric='minisblack')


The next step is to choose a subset of markers from 10x Xenium to transform into the IMC resolution. Importantly, this step converts the Xenium expression matrices in sparse Anndata format into a dense numpy array that can be concatenated to the IMC channel stack for joint visualisation in Rakaia. As such, it is recommended to choose just a subset of markers to view, as >50-75 markers will create a very large output tiff. Here, we just select a few markers associated with B cells to look at. 

In [28]:
type_gene_used = "select_with_dapi"

genes_use = ['CD4', 'CD8A', 'CD19', 'CD22', 'MS4A1']

xenium_transformed_stacks = []

for gene in genes_use:
    raster = spatial_grid_single_marker(adata, gene, 5, True, False)
    raster_transform = warp(
        image=raster,
        inverse_map=tform_1,
        output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
        order=0,
        mode='constant',
        cval=0.0,
        preserve_range=True
    ).astype(np.uint32)
    xenium_transformed_stacks.append(raster_transform)


We also take the DAPI IF image and transform that into the IMC resolution, so that we can visualize the IF nuclei alongside the IMC Iridium nuclear signal. This will allow us to see how well the alignment performed at the single-nuclei level. 

In [29]:
dapi_transformed = warp(
        image=dapi_if,
        inverse_map=np.vstack([twocan_affine, [0,0,1]]),
        output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
        order=0,
        mode='constant',
        cval=0.0,
        preserve_range=True
    ).astype(np.uint32)

xenium_transformed_stacks.append(dapi_transformed)


Now that we have transformed all of the relevant Xenium markers and DAPI IF image into the IMC resolution, we can combine all of the numpy arrays into one stack and output as one tiff file that can be imported into Rakaia. We will also read in a list of the IMC panel markers, and append the additional Xenium and DAPI labels in a txt file, so that we have a list of all of the combined multimodal channels in the order that they appear in the final output tiff. 

This creates the final multimodal stack `full_stack_imc_first_imc_res_xenium_select_with_dapi.tiff` as well as the order marker labels for this tiff in `final_combined_panel.txt` in this directory. 

In [30]:
with open('imc_panel.txt', 'r') as file:
    lines_list = file.read().splitlines()

genes_use = [f"{gene}_xenium" for gene in genes_use] + ['DAPI_IF']

panel_output = lines_list + genes_use

imwrite(
    f"full_stack_imc_first_imc_res_xenium_{type_gene_used}.tiff",
    np.concatenate([imc_stack, np.stack(xenium_transformed_stacks, axis=0)], axis=0).astype(np.float32),
    photometric="minisblack")

with open(f"final_combined_panel.txt", 'w') as file:
    file.write('\n'.join([marker for marker in panel_output]))

